🌐 [English](dx_app.ipynb) | [한국어](dx_app.ko.ipynb) | **日本語** | [中文](dx_app.zh.ipynb)

# DEEPX Tutorial 02 - DX-APP の使い方

この 2 つ目のチュートリアルでは、DX-APP を紹介し、コンパイル済みの DXNN モデルを AI アプリケーションで実行する方法を学びます。

## 学習目標

このチュートリアルを終えると、次のことができるようになります。

- DX-APP が **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) のすべてのモデル**に対してすぐに実行できる C++ と Python のサンプルコードを提供していることを理解し、そのコードを使って任意のモデルを実行する
- DX-APP とは何か、その C++ と Python のサンプルがどのように構成されているかを説明する
- チュートリアルに必要なモデルと動画だけをダウンロードする
- 画像、動画、カメラを入力として、分類、検出、姿勢推定、セグメンテーションのサンプルを実行する
- 同期、非同期、PPU の各実行方式を使い分ける
- 対話型のデモランチャーを起動する
- スタンドアロンの C++ または Python サンプルをエクスポートし、DX-APP ツリーの外で実行する

## DX-APP とは?

**DX-APP** は DEEPX NPU 向けのリファレンスアプリケーションです。その最大の特徴は網羅性です。**[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) に公開されているすべてのモデルに、対応するすぐに実行可能なサンプル**が C++ と Python の両方で用意されており、タスク別(object detection、classification、face detection、pose、segmentation、depth など)に整理されています。Model Zoo のページでモデルを選べば、それを実行するコードはすでに DX-APP に含まれています。DX-APP は開発者がランタイム環境を素早くセットアップするのを助け、
独自の AI アプリケーションを構築・カスタマイズするためのテンプレートとして機能します。

詳細は [dx_app git repo](https://github.com/DEEPX-AI/dx_app/blob/main/README.md) の概要を参照するか、[こちら](https://developer.deepx.ai/download/?id=580)から DX-APP ユーザーガイドをダウンロードしてください!

> **注:** ユーザーガイドをダウンロードするには、まず https://developer.deepx.ai/ にログインする必要があります。

DX-APP のファイル構成を見てみましょう。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** のデモは、最小限のセットアップで DEEPX NPU 上のコンパイル済みモデルを紹介できるよう最適化されています。
各デモは一般的な AI タスクを表しており、画像、動画、またはライブカメラ入力を使って
実行できます。

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP リポジトリの構成:**
```shell
dx_app/
├── src/
│   ├── cpp_example/            # C++ end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Model-specific factories, configs, and entry points
│   │   └── common/             # Shared C++ runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   42 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   16 task-specific sync/async runner pairs
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   13 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader
│   │       ├── trackers/       #   Shared IoU tracker
│   │       ├── third_party/    #   Vendored header dependencies
│   │       └── utility/        #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/         # Python end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Four execution/postprocessing variants per model
│   │   └── common/             # Shared Python runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   41 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   SyncRunner, AsyncRunner, arguments, and run utilities
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   14 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader and schema
│   │       ├── trackers/       #   Shared IoU tracker
│   │       └── utility/        #   Labels, preprocessing, profiling, and geometry helpers
│   ├── postprocess/            # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           └── dx_postprocess/ # pybind11 bindings wrapping src/postprocess/
├── config/
│   ├── model_registry.json     # Model registry — single source of truth
│   ├── test_models.conf        # Test model configuration
│   └── README.md               # Config directory documentation
├── scripts/                    # Developer tools, validation, and helper scripts
├── tests/                      # C++, Python, script, helper, and Windows tests
├── assets/                     # Links to downloaded models and videos
├── sample/                     # Bundled images and task-specific sample data
├── bin/                        # C++ executables generated by the build
├── cmake/                      # x86_64 and aarch64 toolchain files
├── extern/                     # External header dependencies
├── build.sh / build.bat        # Linux and Windows build scripts
├── setup.sh / setup.bat        # Model, video, and dependency setup
├── setup_sample_models.sh      # Download the sample model set only
├── setup_sample_videos.sh      # Download the sample video set only
├── run_demo.sh / run_demo.bat  # Interactive demo launcher
├── run_tc.sh                   # Unified test runner for example tests
├── install.sh                  # Dependency and OpenCV installer
└── docs/                       # Detailed documentation
```
---
**DX-APP は C++ と Python の両方のサンプルを提供します:**

`cpp_example` と `python_example` はどちらも同じタスク優先の構造に従っています。

現在、両言語とも同じ 22 個のタスクディレクトリを提供しています。

- 3d_object_detection/
- attribute_recognition/
- classification/
- depth_estimation/
- embedding/
- face_alignment/
- face_detection/
- hand_detection/
- hand_landmark/
- image_denoising/
- image_enhancement/
- instance_segmentation/
- keypoint_detection/
- obb_detection/
- object_detection/
- object_pose_estimation/
- panoptic_driving_perception/
- pose_estimation/
- ppu/
- reid/
- semantic_segmentation/
- super_resolution/

## 前提条件

- チュートリアル 01 を完了していること: DX-Runtime が `--all` でインストール済み(DX-RT、DX-APP、およびその `venv-dx-runtime` Python 環境)で、NPU が `/dev/dxrt0` として認識されている。
- DX-APP のサンプルモデルと動画。チュートリアル 01 の 3.3 節でダウンロードします。以下の第 1 節では、まだ不足しているものだけを取得します(このチュートリアル用のモデル約 300 MB と、新規ホストでは共有サンプル動画アーカイブ 1.2 GB)。
- グラフィカルなデスクトップセッションは任意です。第 2 節のデモセルはヘルプテキストを表示するかヘッドレスで実行され、第 3 節はスタンドアロンのサンプルをヘッドレスで実行し、保存された画像をノートブック内に表示します。
- `sudo` は不要です。所要時間は約 30 分で、3.3 節の C++ ビルドに 1〜2 分かかります。

次のセルは SDK の場所を特定し、これらの要件を確認してステータス表を表示します。`MISSING` と表示された項目には、それを提供する手順が併記されます。

## 1. モデルと動画の準備

1. `dx_app` ディレクトリに移動します。

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. `setup.sh` で必要なモデルとサンプル動画をダウンロードします。このスクリプトは対話型(どのモデルカテゴリをダウンロードするか尋ねます)なので、セルではなくターミナルで実行します。**File > New > Terminal** を開いて次を実行してください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 次のセルは、同じ 2 つのコマンドを実際のインストールパスで表示します。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. 空きストレージが 29 GB 以上ある場合は、**Enter** を押してすべての Model Zoo モデルをダウンロードします。

5. そうでない場合は、下のコードセルでこのチュートリアルに必要なモデルだけをダウンロードします。チュートリアル 01 の 3.3 節では、このリストを含む 24 個のモデルをすでにダウンロードしています。完了済みであれば、`--no-force` によってセルはすべてのファイルをスキップし、数秒で終了します。後の第 3 節では、使用する 2 つの YOLO26 モデルを同じ方法で追加します。

In [ ]:
# The models used in this tutorial. Tutorial 01 section 3.3 already downloads them;
# this cell only downloads what is still missing (--no-force skips existing files).
SELECTED_MODELS = shlex.join([
    "ResNet50",
    "YoloV7",
    "YoloV7_PPU",
    "yolov8s_pose",
    "yolov8n_seg",
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat "{MODEL_MANIFEST_PATH}" | grep name

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force


6. モデルと動画の両方が期待どおりにダウンロードされたことを確認します。

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. デモの実行

**DX-APP** は、分類、検出、セグメンテーション、姿勢推定など、コンパイル済みモデルを DEEPX NPU 上で実行する方法を示す、すぐに使えるデモアプリケーション群です。

ユーザーは多様な DX-APP のデモを調べることで、アプリケーションの学習と構築ができます。

### 2.1 分類
- **bin/resnet50_async** は、学習済みの画像分類モデルを実行し、最も確率の高いクラスを出力する DX-APP の実行ファイルです。

In [ ]:
%cd {DX_APP_DIR} 

#!ls bin
!./bin/resnet50_async -h
#!./bin/resnet50_async -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg

### 2.2 物体検出
- **bin/yolov\<version>_async** は、YOLO ベースの物体検出モデルを実行し、画像、動画、カメラ内の物体を検出する DX_APP のデモバイナリです。

In [ ]:
%cd {DX_APP_DIR} 

# Find other yolo based reference excutable files
!ls ./bin/yolo*

#### 実行方式

DX-APP は、アプリケーションの要件に合わせてさまざまな実行方式を提供します。

| 方式 | 主な特徴 | 推奨用途 |
|---|---|---|
| 同期 | 各フレームを順番に処理する | シンプルなフローとレイテンシの確認 |
| 非同期 | パイプラインの各ステージを重ねて実行する | より高いスループット |
| PPU | 対応する後処理を NPU 側にオフロードする | ホスト CPU の負荷軽減 |

- **同期アプリケーション**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_sync -h

# Image input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **非同期アプリケーション**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_async -h

# Image input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (This IP address only works in Korea)
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU と非 PPU の比較**

以下の 2 つのセルは、同じ 7 秒のクリップを非 PPU 版と PPU 版の YOLOv7 モデルで実行します。`SHOW_WINDOW = False`(既定)の場合は `--no-display` を付けてヘッドレスで実行し、FPS のサマリーだけを表示するため、ノートブックがブロックされることはありません。デスクトップセッションでは `SHOW_WINDOW = True` に設定すると、DX-APP のウィンドウで検出結果と FPS カウンターを確認できます。2 つの FPS の行を比較してください。PPU モデルは候補のフィルタリングとクラス選択を NPU 側に移しています。

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


### 2.3 姿勢推定
- **bin/yolov8s_pose_sync** は、姿勢推定モデルを実行し、画像、動画、カメラ内の人物を検出してキーポイントを推定する DX_APP のデモバイナリです。

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8s_pose_sync -h

# Image input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov

# Camera input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 セグメンテーション
- **bin/yolov8n_seg_async** は、インスタンスセグメンテーションモデルを実行し、画像、動画、カメラに対してピクセル単位のクラスラベルを生成する DX_APP のデモバイナリです。

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8n_seg_async -h

# Image input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4

# Camera input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 デモスクリプト

`run_demo.sh` は、C++ と Python 両方のデモを起動できる対話型ランチャーです。

対話型なので、ターミナル(**File > New > Terminal**)で実行してください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

次のセルは、同じコマンドを実際のパスで表示します。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


> ![](assets/sc-run_demo.png)

このランチャーは、DX-APP のタスクカテゴリ全体にわたって、すぐに実行できる 23 個のデモを提供します。

`0` の Object Detection を選択すると、次の実行オプションが利用できます。
> ![](assets/sc-run_demo2.png)

オプション [1, 2] は C++ ベースのアプリケーションを、オプション [3〜6] は Python ベースのアプリケーションを実行します。

> **注:** Python ベースのアプリケーションを実行するには、`dx-engine` がインストールされた Python 仮想環境を有効化してください。

例:
> ![](assets/sc-run_demo3.png)

### 2.6 C++ と Python で書かれたサンプルアプリケーションの抽出

`scripts/dx_tool.sh` は、サンプルの管理とエクスポートを行う対話型のターミナルツールです。その `extract` メニューは、1 つのモデルのサンプルを共有の `common/` ランタイム層とともに、DX-APP ツリーの**外**でビルドして実行できる自己完結型のパッケージにコピーします。ターミナル(File > New > Terminal)で実行してください。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

次のセルは、同じコマンドを実際のパスで表示します。メニューが表示されたら、その下の手順に従ってください。

このメニューは内部で `scripts/extract_model_package.sh` を呼び出しています。第 3 節ではこのスクリプトを直接使用します。モデルと言語を引数として受け取るため、ノートブックのセル内で実行できるからです。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


`dx_tool.sh` を実行すると、次のメニューが表示されます。 
- `3` を選ぶと、関心のある利用可能なモデルの一覧を確認できます。
- `2` を選ぶと、サンプルコードを抽出します。
- プログラミング言語を選択します。(C++、Python)
- アプリケーションのカテゴリとモデルを入力します。(例: `object_detection/yolo26s`)

> ![](assets/sc-extract.png)

数秒後、サンプルコードが `output` ディレクトリに生成されます。

> ![](assets/sc-extract2.png)

## 3. 抽出したスタンドアロンサンプルの実行

2.6 節では、サンプルを対話的にエクスポートしました。この節では、同じことをノートブックから `scripts/extract_model_package.sh` で行い、エクスポートしたパッケージを **DX-APP ツリーの外**で、自分のプロジェクトで行うのとまったく同じようにビルドして実行します。3 つの例で、2 つの言語と、Python の 4 つのバリアントのうち 2 つを扱います。

| 例 | モデル(タスク) | 言語とバリアント | 示す内容 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26s` | Python、`*_sync.py` | 任意の Model Zoo モデルを実行する最も簡単な方法 |
| 3.3 | `object_detection/yolo26s` | C++、`*_sync` | エクスポートした C++ パッケージを CMake でビルドする |
| 3.4 | `pose_estimation/yolo26s_pose` | Python、`*_async_cpp_postprocess.py` | 別のタスクと、最もスループットの高いバリアント |

エクスポートされた Python サンプルはすべて `dx_engine` と `dx_postprocess` をインポートします。これらは DX-Runtime が `dx-runtime/venv-dx-runtime` にインストールしたものです。そのため、セルはそのインタープリターを明示的に呼び出します。エクスポートされたサンプルにはサンプル画像が含まれていないため、セルは `dx_app/sample/img` の画像を渡してヘッドレス(`--no-display --save`)で実行し、保存された結果をノートブック内に表示します。

### 3.1 パッケージのエクスポート

まずスクリプト自身のヘルプから始めます。次のセルで使用する引数が一覧されます。モデルは `<task>/<model>`、`--lang cpp|py|both`、そして `--output-dir` です。

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


`--output-dir` を指定しない場合、スクリプトは `dx_app` 内のその場でパッケージを準備します。指定した場合、パッケージは `<dir>/cpp/<task>/<model>` と `<dir>/py/<task>/<model>` にコピーされます。次のセルは、3 つのパッケージをこのチュートリアルの `workspace/` ディレクトリ(git では無視されます)にエクスポートします。

In [ ]:
from IPython.display import Image, display

STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26s yolo26s-pose --no-force

# Export: object detection in both languages, pose estimation in Python.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26s --lang both --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26s_pose --lang py --output-dir "{STANDALONE_DIR}"

!tree -L 3 "{STANDALONE_DIR}"


### 3.2 例 1: Python スタンドアロン、物体検出

`yolo26s_sync.py` は、純粋な Python の後処理で 1 フレームずつ処理する、プレーンな同期バリアントです。最初に読むべきバリアントです。引数は DX-APP のすべての Python サンプルで共通です。`--model`、入力オプションのいずれか 1 つ(`--image`、`--video`、`--camera`、`--rtsp`)、そしてヘッドレス実行用の `--no-display --save` です。

```bash
cd <workspace>/standalone/py/object_detection/yolo26s
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26s"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_sync-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.3 例 2: C++ スタンドアロン、物体検出

エクスポートされた C++ パッケージには、`yolo26s_sync.cpp`、`yolo26s_async.cpp`、共有の `common/` 層、そしてインストール済みの DX-RT にリンクする生成済みの `CMakeLists.txt` が含まれています。ビルドは標準的な CMake の手順で、DX-APP ツリーを参照するものは何もありません。

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26s
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26s_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

C++ サンプルではオプションが `-m/--model_path` と `-i/--image_path` と表記される点に注意してください。`./yolo26s_sync --help` で一覧できます。ビルドには 1〜2 分かかります。

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26s"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26s_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("Yolo26s_sync-*/output.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.4 例 3: Python スタンドアロン、姿勢推定、C++ 後処理による非同期実行

すべての Python サンプルには 4 つのバリアントがあります。ファイル名を見れば、どれを実行しているかがわかります。

| バリアント | 後処理 | スレッド方式 | 用途 |
|---|---|---|---|
| `*_sync.py` | 純粋な Python | 同期 | ロジックの学習とデバッグ |
| `*_async.py` | 純粋な Python | 非同期 | 性能向上の第一歩 |
| `*_sync_cpp_postprocess.py` | C++ バインディング(`dx_postprocess`) | 同期 | CPU 負荷の高い後処理 |
| `*_async_cpp_postprocess.py` | C++ バインディング(`dx_postprocess`) | 非同期 | 最大 FPS。デプロイに使うバリアント |

この例では、別のタスクと最速のバリアントに切り替えます。コマンドの形は変わりません。変わるのはディレクトリ、スクリプト名、モデルだけです。

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26s_pose
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_pose_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26s_pose"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_pose_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_pose_async_cpp_postprocess-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.5 試してみよう: Model Zoo の任意のモデル

3 つの例は 1 つのパターンに従っており、これは 352 個すべてのモデルに当てはまります。

1. [Model Zoo](https://developer.deepx.ai/modelzoo/) のページでモデルを選び、`./scripts/dx_tool.sh search <keyword>` または `./scripts/dx_tool.sh list <category>` でその DX-APP 上の名前を調べます。
2. ダウンロードします: `./setup.sh --models <name> --no-force`
3. エクスポートします: `scripts/extract_model_package.sh <task>/<name> --lang py --output-dir <dir>`
4. 実行します: `<dir>/py/<task>/<name>/<name>_sync.py --model <file>.dxnn --image <image>`

次のセルは `config/model_registry.json` でモデルを検索し、そのモデル用の 4 つのコマンドを表示します。`MODEL_NAME` を試したいモデルに変更し、表示されたコマンドをセルまたはターミナルで実行してください。例えば `scrfd500m`(顔検出、`sample_face.jpg` を使用)や `resnet50`(分類、`sample/ILSVRC2012` の画像を使用)などです。

In [ ]:
import json

MODEL_NAME = "scrfd500m"   # change me: any model_name from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if e["model_name"] == MODEL_NAME), None)
if entry is None:
    raise KeyError(f"{MODEL_NAME} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {MODEL_NAME}")

task, dxnn = entry["add_model_task"], entry["dxnn_file"]
print(f"{MODEL_NAME}: task={task}, model file={dxnn}\n")
print(f'cd "{DX_APP_DIR}" && bash setup.sh --models {MODEL_NAME} --no-force')
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{MODEL_NAME} --lang py --output-dir "{STANDALONE_DIR}"')
print(f'cd "{STANDALONE_DIR}/py/{task}/{MODEL_NAME}" && "{DX_RUNTIME_PYTHON}" {MODEL_NAME}_sync.py --model "{MODEL_DIR / dxnn}" --image <an image under {SAMPLE_IMG_DIR}> --no-display --save --save-dir "{OUTPUT_DIR}"')


## 4. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| 実行ファイルがない | `./bin/yolov7_async: No such file or directory` | DX-APP がビルドされていないか、そのターゲットなしでビルドされた | ターミナルで `dx_app` 内の `./build.sh`(または `./build.sh --target yolov7_async`)を実行する |
| モデルを読み込めない | `assets/models` 配下のパスを含む `Failed to load model` | モデルがダウンロードされていない | 第 1 節のダウンロードセルをそのモデル名で実行し、再実行する |
| 出力ウィンドウが表示されない | `cannot open display`、またはウィンドウなしでセルが終了する | グラフィカルセッションがない(SSH またはヘッドレスホスト) | デスクトップセッションで実行するか、画像入力(`-i`)を使って保存結果を確認する |
| カメラを開けない | `Could not open camera 0` | `/dev/video0` がないか、ユーザーが `video` グループに属していない | `v4l2-ctl --list-devices` を確認し、`sudo usermod -aG video $USER` を実行して再ログインする |
| `run_demo.sh` が勝手にデモを開始する | 10 秒後にオプション 0 が実行される | ランチャーには 10 秒の入力タイムアウトがある | メニューが表示されたらすぐにオプション番号を入力する |
| エクスポートした Python サンプルで `dx_engine` をインポートできない | `ModuleNotFoundError: No module named 'dx_engine'` | DX-Runtime の環境ではなく Jupyter の Python で実行された | 第 3 節のセルと同様に `<dx-runtime>/venv-dx-runtime/bin/python` を使う |
| エクスポートした C++ パッケージの構成に失敗する | `Could not find a package configuration file provided by "dxrt"` | DX-RT がインストールされていないか、カスタムプレフィックス配下にある | DX-Runtime をインストールする(チュートリアル 01 の第 3 節)か、`cmake` の行に `-DCMAKE_PREFIX_PATH=<prefix>` を追加する |
| エクスポートしたサンプルが入力を見つけられない | `Input file not found: sample/img/...` | パッケージにサンプル画像が含まれていない | `dx_app/sample/img` のファイルを `--image`(Python)または `-i`(C++)で渡す |

## 5. まとめ

### 5.1 完了した DX-APP ワークフロー

```text
Load SDK paths
       │
       ▼
Prepare selected models and videos
       │
       ▼
Run C++ or Python examples
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Use run_demo.sh or export a standalone example
```

### 5.2 学習ダッシュボード

| 領域 | できるようになったこと |
|---|---|
| SDK パス | `config.json` から `DX_APP_DIR` と関連パスを読み込む |
| モデルのセットアップ | 演習に必要なモデルと動画だけをダウンロードする |
| サンプルの選択 | タスク、モデル、言語、実行方式でサンプルを探す |
| 実行方式 | 目的に応じて同期、非同期、PPU の実行方式を選ぶ |
| デモランチャー | `run_demo.sh` でインストール済みのサンプルを起動する |
| スタンドアロンコード | `scripts/extract_model_package.sh` で任意のモデルのサンプルをエクスポートし、DX-APP の外でビルドして実行する |
| Model Zoo の網羅性 | `config/model_registry.json` または `dx_tool.sh search` で任意の Model Zoo モデルに対応する DX-APP サンプルを見つける |

### 5.3 完了チェックリスト

- [ ] DEEPX NPU が DX-RT から認識されていることを確認した
- [ ] 必要な DX-APP のモデルと動画アセットを準備した
- [ ] エンドツーエンドの推論サンプルを少なくとも 1 つ実行した
- [ ] 利用可能な実行方式を比較した
- [ ] 対象タスクのリファレンス実装を見つけた
- [ ] `run_demo.sh` を開いた
- [ ] スタンドアロンの Python と C++ サンプルを DX-APP ツリーの外でエクスポート、ビルド、実行した
- [ ] 自分で選んだ Model Zoo モデルを実行した

> **次へ:** チュートリアル 03 に進み、学習済みモデルを調整して DXNN にコンパイルし、DX-APP のランタイム設定に統合してみましょう。